# Práctica 8: Detección de personas / reconocimiento facial
Vas a ampliar la base de datos de rostros y optimizar la frecuencia de reconocimiento. Si te trabás, repasá la lección en `../notebooks/8_deteccion_personas.ipynb`.

### Ejercicio 1: Agregar una nueva persona a la base de datos
Creá una subcarpeta nueva dentro de `../db` con el nombre de una persona y agregale 2 o 3 fotos, para que `DeepFace.find` pueda reconocerla.

In [ ]:
import os  # Qué: importa el módulo estándar de Python para interactuar con el sistema de archivos. Por qué: se necesita para construir rutas de forma portable y crear carpetas nuevas dentro de `../db`.

# TODO: definí el nombre de la nueva persona que vas a agregar a la base de datos
# Pista: es el string que va a funcionar como nombre de la subcarpeta y, más adelante, como el nombre que la función `reconocimiento()` extrae de la ruta del archivo (mirá cómo se parsea `nombre` en notebooks/8_deteccion_personas.ipynb).
nombre_nueva_persona = ""

# TODO: creá la carpeta ../db/<nombre_nueva_persona> si no existe usando
# os.makedirs, y copiá ahí 2 o 3 fotos de esa persona
# Pista: DeepFace.find espera que `db_path` tenga una subcarpeta por persona con sus fotos adentro; `os.makedirs` puede fallar si la carpeta ya existe, revisá qué parámetro evita ese error.
ruta_nueva_persona = os.path.join("../db", nombre_nueva_persona)  # Qué: arma la ruta completa a la carpeta de la nueva persona. Cómo: os.path.join concatena los segmentos usando el separador correcto del sistema operativo (evita hardcodear '/' o '\\'). Por qué: es la ruta que se va a usar en el TODO de arriba para crear la carpeta.

### Ejercicio 2: Ajustar la frecuencia de reconocimiento (throttle)
Llamar a `reconocimiento()` en cada frame es costoso. Modificá el loop para que solo se ejecute cada N frames.

In [ ]:
from deepface import DeepFace  # Qué: importa DeepFace, la librería que hace la búsqueda/comparación de rostros contra la base de datos de fotos. Por qué: se reutiliza la misma función `find` que en la lección.
import cv2  # Qué: importa OpenCV. Por qué: se usa para capturar video de la webcam y mostrar/anotar los cuadros.


def reconocimiento(frame):  # Qué: función que intenta identificar a la persona presente en `frame`. Por qué: encapsula la búsqueda facial y el manejo de errores en un único punto reutilizable.
    try:
        recognition = DeepFace.find(frame, db_path='../db', model_name='VGG-Face', silent=True)  # Qué: busca el rostro de `frame` dentro de la base de datos `../db`. Cómo: genera el embedding facial del cuadro con el modelo VGG-Face y lo compara contra los embeddings de las fotos organizadas por subcarpeta de persona. Por qué: reutiliza fotos ya etiquetadas por carpeta en vez de entrenar un clasificador propio.
        recognition2 = recognition[0]['identity'][0]  # Qué: toma la ruta de la imagen más parecida (mejor coincidencia) del resultado. Por qué: de ahí se extrae el nombre de la persona en el siguiente paso.
        nombre = recognition2.split('\\')[1].split('/')[0]  # Qué: extrae el nombre de la persona a partir de la ruta del archivo. Cómo: la ruta tiene forma `../db\\NombrePersona\\foto.jpg`; se toma el segundo segmento tras separar por `\\` (el nombre de la subcarpeta). Por qué: el nombre está codificado como el nombre de la carpeta dentro de `db`, no como un campo aparte.
        return nombre  # Qué: devuelve el nombre identificado. Por qué: es el resultado útil cuando el reconocimiento tuvo éxito.
    except ValueError:  # Qué: captura el error que DeepFace lanza cuando NO detecta ningún rostro en `frame`. Por qué: sin este catch, el loop principal se rompería en cada cuadro sin cara visible.
        return 'Rostro No Detectado'  # Qué: valor de reemplazo cuando no hay cara.
    except KeyError:  # Qué: captura el error cuando SÍ hay un rostro pero no hubo coincidencias en la base de datos (el resultado viene vacío y acceder a `['identity'][0]` falla). Por qué: es un fallo distinto al de ValueError (cara detectada pero desconocida, vs. ninguna cara detectada), por eso se capturan por separado aunque el resultado final sea el mismo.
        return 'Rostro No Detectado'  # Qué: mismo mensaje de fallback para mantener consistencia con el resto del programa.


vid = cv2.VideoCapture(0)  # Qué: abre la webcam por defecto (índice 0). Por qué: fuente de video en vivo sobre la que se ejecuta el reconocimiento.
cv2.namedWindow('Reconocimiento Facial', cv2.WINDOW_NORMAL)  # Qué: crea la ventana de forma explícita y redimensionable (WINDOW_NORMAL) antes de usar imshow.

mensaje = ''  # Qué: guarda el último resultado de reconocimiento calculado, para reutilizarlo en los frames donde no se vuelve a llamar a `reconocimiento()`. Por qué: es la base del throttle que se pide implementar en este ejercicio.
contador_frames = 0  # Qué: contador que lleva la cuenta de cuántos cuadros pasaron desde que arrancó el loop. Por qué: permite decidir, junto con `frecuencia_reconocimiento`, en qué frames corresponde ejecutar el reconocimiento costoso.

# TODO: definí cada cuántos frames se va a llamar a reconocimiento() (por ejemplo cada 5)
# Pista: es un número entero N; con N=5, el reconocimiento se ejecutaría 1 de cada 5 cuadros, reduciendo la carga de cómputo sin dejar de actualizar el `mensaje` periódicamente.
frecuencia_reconocimiento = None

while True:  # Qué: procesa la webcam en vivo, cuadro por cuadro, hasta que el usuario salga.
    ret, frame = vid.read()  # Qué: captura el siguiente cuadro de la webcam. Cómo: `ret` indica éxito, `frame` es la imagen BGR actual.

    # TODO: llamá a reconocimiento(frame) solo cuando contador_frames sea múltiplo
    # de frecuencia_reconocimiento (usá el operador %) y actualizá `mensaje`;
    # en el resto de los frames reutilizá el último `mensaje` calculado
    # Pista: DeepFace.find es costoso porque corre un modelo de deep learning en cada llamada; el objetivo es no ejecutarlo en cada frame sino cada N frames, dejando que `mensaje` conserve su último valor en los frames intermedios (similar en espíritu al patrón de `mensajeAnterior` de la lección, pero aplicado a throttling en vez de a evitar repetir el saludo).

    cv2.putText(frame, mensaje, (0, 115), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0))  # Qué: dibuja el último `mensaje` reconocido sobre el cuadro actual. Cómo: fuente FONT_HERSHEY_SIMPLEX, escala 1, color verde. Por qué: aunque no se recalculó el reconocimiento en este frame, se sigue mostrando el último resultado conocido para que el video no "parpadee" sin etiqueta.
    cv2.imshow('Reconocimiento Facial', frame)  # Qué: muestra el cuadro anotado en la ventana. Por qué: salida visual en vivo del reconocimiento.

    contador_frames += 1  # Qué: incrementa el contador de cuadros procesados. Por qué: debe avanzar en cada iteración para que el cálculo de "múltiplo de frecuencia_reconocimiento" del TODO de arriba tenga sentido.

    if cv2.waitKey(1) == ord('q'):  # Qué: espera 1 ms por una tecla y compara con 'q'. Por qué: permite salir del loop de forma controlada.
        break  # Qué: corta el bucle infinito. Por qué: única salida controlada del loop.

vid.release()  # Qué: libera el dispositivo de cámara. Por qué: evita bloquear la webcam para otros procesos.
cv2.destroyAllWindows()  # Qué: cierra todas las ventanas abiertas de OpenCV. Por qué: limpieza de recursos de GUI al terminar.